# Código para crear la base de datos desde el inicio

In [ ]:
import requests
import json
import re
import sqlite3


url = 'https://smn.conagua.gob.mx/tools/PHP/ConsultaCapas.php?tipo=ENCS&clave=Act'

response = requests.get(url)
print("Status code:", response.status_code)
print(response.text)



Status code: 200
[{"id_estacion":"1","nombre":"AGUASCALIENTES (OBS)","latitud":"21.85027778","longitud":"-102.2908333","icono":"enc"},{"id_estacion":"2","nombre":"CALVILLO (SMN)","latitud":"21.88333333","longitud":"-102.7188889","icono":"encError"},{"id_estacion":"3","nombre":"CA\u00d1ADA HONDA","latitud":"22.00083333","longitud":"-102.1988889","icono":"enc"},{"id_estacion":"4","nombre":"PRESA EL NIAGARA","latitud":"21.78055556","longitud":"-102.3716667","icono":"enc"},{"id_estacion":"5","nombre":"EL TULE (SMN)","latitud":"22.08166667","longitud":"-102.0913889","icono":"encError"},{"id_estacion":"6","nombre":"JESUS MARIA (SMN)","latitud":"21.96222222","longitud":"-102.3447222","icono":"encError"},{"id_estacion":"7","nombre":"PUERTO DE LA CONCEPCION","latitud":"22.20277778","longitud":"-102.135","icono":"enc"},{"id_estacion":"8","nombre":"LA LABOR (SMN)","latitud":"21.96222222","longitud":"-102.6947222","icono":"encError"},{"id_estacion":"9","nombre":"LA TINAJA","latitud":"22.16444444",

In [ ]:
try:
    data = response.json()
    print("JSON Data:", data)
except json.JSONDecodeError as e:
    print("Failed to decode JSON:", e)


def encontrar_ids(id_estacion: int) ->  tuple:
  """
  Encuentra el id de la estación correspondiente al id del servidor
  y encuentra la clave para el estado en el que se encuentra la estación

  Estos se usan para completar los urls de los datos por estación
  """
  url = 'https://smn.conagua.gob.mx/tools/PHP/GetInfoENCS.php?id_estacion='+str(id_estacion)
  response = requests.get(url)
  try:
    key, id = re.findall(r'([a-z]*)..dia([0-9]*).txt"',response.text)[0]
    return key, id
  except:
    print('No hay datos de Climatología diaria')
    return None, None

def encontrar_url_diario(key: str, id: str):
  url = 'https://smn.conagua.gob.mx/tools/RESOURCES/Normales_Climatologicas/Diarios/'+key+'/dia'+id+'.txt'
  return url

def metadata(key,id):
  url = 'https://smn.conagua.gob.mx/tools/RESOURCES/Normales_Climatologicas/Diarios/'+key+'/dia'+id+'.txt'
  response = requests.get(url)
  metadata_start = response.text.find('NOMBRE')
  metadata_end = response.text.find('msnm')
  aux = response.text[metadata_start:metadata_end]
  estado = re.findall(r'ESTADO\s*:\s*(.*)', aux)[0].strip()
  nombre = re.findall(r'NOMBRE\s*:\s*(.*)', aux)[0].strip()
  municipio = re.findall(r'MUNICIPIO\s*:\s*(.*)', aux)[0].strip()
  situacion = re.findall(r'SITUACIÓN\s*:\s*(.*)', aux)[0].strip()
  latitud = re.findall(r'LATITUD\s*:\s*(.*)\s°', aux)[0].strip()
  longitud = re.findall(r'LONGITUD\s*:\s*(.*)\s°', aux)[0].strip()
  altitud = re.findall(r'ALTITUD\s*:\s*(.*)', aux)[0].strip()
  return estado, nombre, municipio, situacion, latitud, longitud, altitud


estado_previo = ''
id_estado = 0

conn = sqlite3.connect('estacionesbasededatos.sqlite3')
cursor = conn.cursor()

cursor.execute('''
    CREATE TABLE IF NOT EXISTS estaciones (
        id INTEGER PRIMARY KEY,
        real_id TEXT NOT NULL,
        id_estado INTEGER NOT NULL,
        nombre TEXT NOT NULL,
        municipio TEXT NOT NULL,
        situacion TEXT NOT NULL,
        latitud REAL,
        longitud REAL,
        altitud REAL,
        state_key TEXT NOT NULL,
        FOREIGN KEY (id_estado) REFERENCES estados (id)
    )
''')

cursor.execute('''
    CREATE TABLE IF NOT EXISTS estados (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre TEXT UNIQUE NOT NULL
    )
''')

for id in range(1, len(data) - 1):
    state_key, real_id = encontrar_ids(id)
    if state_key is None:
        continue

    estado, nombre, municipio, situacion, latitud, longitud, altitud = metadata(state_key, real_id)

    if estado != estado_previo:
        estado_previo = estado

        cursor.execute('''
            INSERT OR IGNORE INTO estados (nombre)
            VALUES (?)
        ''', (estado,))
        conn.commit()

        cursor.execute('''
            SELECT id FROM estados WHERE nombre = ?
        ''', (estado,))
        id_estado = cursor.fetchone()[0]

        print(f"Estado: {estado}, id_estado: {id_estado}")
        print('NUEVO ESTADO-----------------------')

    cursor.execute('''
        INSERT OR IGNORE INTO estaciones (id, real_id, id_estado, nombre, municipio, situacion, latitud, longitud, altitud, state_key)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (id, real_id, id_estado, nombre, municipio, situacion, latitud, longitud, altitud, state_key))
    print('Estación guardada:'+str(id))

conn.commit()

conn.close()



Streaming output truncated to the last 5000 lines.
Estación guardada:570
Estación guardada:571
Estación guardada:572
Estación guardada:573
Estación guardada:574
Estación guardada:575
Estación guardada:576
Estación guardada:577
Estación guardada:578
Estación guardada:579
Estado: COLIMA, id_estado: 10
NUEVO ESTADO-----------------------
Estación guardada:580
Estación guardada:581
Estación guardada:582
Estación guardada:583
Estación guardada:584
Estación guardada:585
Estación guardada:586
Estación guardada:587
Estación guardada:588
Estación guardada:589
Estación guardada:590
Estación guardada:591
Estación guardada:592
Estación guardada:593
Estación guardada:594
Estación guardada:595
Estación guardada:596
Estación guardada:597
Estación guardada:598
Estación guardada:599
Estación guardada:600
Estación guardada:601
Estación guardada:602
Estación guardada:603
Estación guardada:604
Estación guardada:605
Estación guardada:606
Estación guardada:607
Estación guardada:608
Estación guardada:609
Est

In [ ]:
def data_diaria(text):
    """Extrae la información"""
    daily_records = []
    lines = text.splitlines()
    data_start = False

    for line in lines:
      # detecta el inicio de la tabla e inicia la extracción en la siguiente línea
        if "°C" in line:
            data_start = True
            continue

        if data_start:
            parts = line.split()
            if len(parts) == 5:
                fecha = parts[0]
                precip = None if parts[1] == 'NULO' else float(parts[1])
                evap = None if parts[2] == 'NULO' else float(parts[2])
                tmax = None if parts[3] == 'NULO' else float(parts[3])
                tmin = None if parts[4] == 'NULO' else float(parts[4])
                daily_records.append((fecha, precip, evap, tmax, tmin))
    return daily_records



conn = sqlite3.connect('estacionesbasededatos.sqlite3')
cursor = conn.cursor()

cursor.execute('SELECT  state_key, real_id, id FROM estaciones')
ids = cursor.fetchall()

cursor.execute('''
      CREATE TABLE IF NOT EXISTS registros_diarios (
          id INTEGER PRIMARY KEY AUTOINCREMENT,
          estacion_id INTEGER,
          fecha TEXT,
          precip REAL,
          evap REAL,
          tmax REAL,
          tmin REAL,
          FOREIGN KEY (estacion_id) REFERENCES estaciones (id),
          UNIQUE (estacion_id, fecha)
      )
  ''')

for id in ids:
  try:
    print(encontrar_url_diario(id[0],id[1]))
    response = requests.get(encontrar_url_diario(id[0],id[1]))
    print(response.status_code)
    text = response.text
    daily_records = data_diaria(text)
    for record in daily_records:
      cursor.execute('''
          INSERT INTO registros_diarios (estacion_id, fecha, precip, evap, tmax, tmin)
          VALUES (?, ?, ?, ?, ?, ?)
      ''', (id[2], *record))
    print(str(id[2])+' station saved')

  except:
    print('No hay datos de Climatología diaria')

conn.commit()
conn.close()








https://smn.conagua.gob.mx/tools/RESOURCES/Normales_Climatologicas/Diarios/ags/dia01001.txt
200
1 station saved
https://smn.conagua.gob.mx/tools/RESOURCES/Normales_Climatologicas/Diarios/ags/dia01003.txt
200
2 station saved
https://smn.conagua.gob.mx/tools/RESOURCES/Normales_Climatologicas/Diarios/ags/dia01004.txt
200
3 station saved
https://smn.conagua.gob.mx/tools/RESOURCES/Normales_Climatologicas/Diarios/ags/dia01005.txt
200
4 station saved
https://smn.conagua.gob.mx/tools/RESOURCES/Normales_Climatologicas/Diarios/ags/dia01006.txt
200
5 station saved
https://smn.conagua.gob.mx/tools/RESOURCES/Normales_Climatologicas/Diarios/ags/dia01007.txt
200
6 station saved
https://smn.conagua.gob.mx/tools/RESOURCES/Normales_Climatologicas/Diarios/ags/dia01008.txt
200
7 station saved
https://smn.conagua.gob.mx/tools/RESOURCES/Normales_Climatologicas/Diarios/ags/dia01009.txt
200
8 station saved
https://smn.conagua.gob.mx/tools/RESOURCES/Normales_Climatologicas/Diarios/ags/dia01010.txt
200
9 statio

#Para guardr la bas ede datos en el drive, luego proceder a guardarla manualmente en la carpeta de trabaj0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp estacionesbasededatos.sqlite3 /content/drive/MyDrive/

ValueError: mount failed

# Actualizar cada 10 días
Última actualización: 27 de Diciembre

In [ ]:
from google.colab import drive
import requests
import json
import re
import sqlite3


drive.mount('/content/drive')

# Verificar que el path sea el adecuado en la carpeta de trabajo
db_path = '/content/drive/My Drive/Colab Notebooks/scrapper/estacionesbasededatos.sqlite3'

def encontrar_url_diario(key: str, id: str):
  url = 'https://smn.conagua.gob.mx/tools/RESOURCES/Normales_Climatologicas/Diarios/'+key+'/dia'+id+'.txt'
  return url

def data_diaria_update(text):
    """Extrae la información"""
    daily_records = []
    lines = text.splitlines()
    last_10_lines = lines[-10:]


    for line in last_10_lines:
        parts = line.split()
        if len(parts) == 5:
            fecha = parts[0]
            precip = None if parts[1] == 'NULO' else float(parts[1])
            evap = None if parts[2] == 'NULO' else float(parts[2])
            tmax = None if parts[3] == 'NULO' else float(parts[3])
            tmin = None if parts[4] == 'NULO' else float(parts[4])
            daily_records.append((fecha, precip, evap, tmax, tmin))
    return daily_records



conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute('SELECT  state_key, real_id, id FROM estaciones')
ids = cursor.fetchall()


for id in ids:
  try:
    print(encontrar_url_diario(id[0],id[1]))
    response = requests.get(encontrar_url_diario(id[0],id[1]))
    print(response.status_code)
    text = response.text
    daily_records = data_diaria_update(text)
    for record in daily_records:
      cursor.execute('''
          INSERT OR IGNORE INTO registros_diarios (estacion_id, fecha, precip, evap, tmax, tmin)
          VALUES (?, ?, ?, ?, ?, ?)
      ''', (id[2], *record))
    print(str(id[2])+' station saved')

  except:
    print('No hay datos de Climatología diaria')

conn.commit()
conn.close()

MessageError: Error: credential propagation was unsuccessful

# Obtener datos diarios por estado

In [ ]:
from google.colab import drive
import requests
import json
import re
import sqlite3


drive.mount('/content/drive')

db_path = '/content/drive/My Drive/Colab Notebooks/scrapper/estacionesbasededatos.sqlite3'

def encontrar_url_diario(key: str, id: str):
  url = 'https://smn.conagua.gob.mx/tools/RESOURCES/Normales_Climatologicas/Diarios/'+key+'/dia'+id+'.txt'
  return url

def data_diaria_show(text):
    """Extrae la información"""
    daily_records = []
    lines = text.splitlines()
    line = lines[-1]
    parts = line.split()
    if len(parts) == 5:
        fecha = parts[0]
        precip = None if parts[1] == 'NULO' else float(parts[1])
        evap = None if parts[2] == 'NULO' else float(parts[2])
        tmax = None if parts[3] == 'NULO' else float(parts[3])
        tmin = None if parts[4] == 'NULO' else float(parts[4])
        daily_records.append((fecha, precip, evap, tmax, tmin))
    return daily_records


conn = sqlite3.connect(db_path)
cursor = conn.cursor()

################### SOLO MAYÚSCULAS
estado_a_buscar = 'JALISCO'


cursor.execute('''
    SELECT estaciones.state_key, estaciones.real_id, estaciones.id, estaciones.nombre
    FROM estaciones
    JOIN estados ON estaciones.id_estado = estados.id
    WHERE estados.nombre = ? AND estaciones.situacion = ?
''', (estado_a_buscar, 'OPERANDO'))
ids = cursor.fetchall()



print('FECHA		PRECIP	EVAP	TMAX	TMIN')
print('.                (mm)	(mm)	(°C )	(°C)')
for id in ids:
  try:
    response = requests.get(encontrar_url_diario(id[0],id[1]))
    text = response.text
    daily_records = data_diaria_show(text)
    print()
    print(id[3],daily_records)

  except:
    print('No hay datos de Climatología diaria')

conn.close()

MessageError: Error: credential propagation was unsuccessful

# Obtener datos diarios por municipio

In [ ]:
from google.colab import drive
import requests
import json
import re
import sqlite3


drive.mount('/content/drive')

db_path = '/content/drive/My Drive/Colab Notebooks/scrapper/estacionesbasededatos.sqlite3'

def encontrar_url_diario(key: str, id: str):
  url = 'https://smn.conagua.gob.mx/tools/RESOURCES/Normales_Climatologicas/Diarios/'+key+'/dia'+id+'.txt'
  return url

def data_diaria_show(text):
    """Extrae la información"""
    daily_records = []
    lines = text.splitlines()
    line = lines[-1]
    parts = line.split()
    if len(parts) == 5:
        fecha = parts[0]
        precip = None if parts[1] == 'NULO' else float(parts[1])
        evap = None if parts[2] == 'NULO' else float(parts[2])
        tmax = None if parts[3] == 'NULO' else float(parts[3])
        tmin = None if parts[4] == 'NULO' else float(parts[4])
        daily_records.append((fecha, precip, evap, tmax, tmin))
    return daily_records


conn = sqlite3.connect(db_path)
cursor = conn.cursor()

################### SOLO MAYÚSCULAS
#### El nombre del municipio tiene que estar completo, sin acentos, por ejemplo:
#### municipio_a_buscar = 'SAN PEDRO TLAQUEPAQUE'
#### municipio_a_buscar = 'TLAJOMULCO DE ZUÑIGA'
municipio_a_buscar = 'GUADALAJARA'

cursor.execute('''
    SELECT state_key, real_id, id, nombre
    FROM estaciones
    WHERE municipio = ?
''', (municipio_a_buscar,))
ids = cursor.fetchall()



print('FECHA		PRECIP	EVAP	TMAX	TMIN')
print('.                (mm)	(mm)	(°C )	(°C)')
for id in ids:
  try:
    response = requests.get(encontrar_url_diario(id[0],id[1]))
    text = response.text
    daily_records = data_diaria_show(text)
    print()
    print(id[3],daily_records)

  except:
    print('No hay datos de Climatología diaria')

conn.close()

MessageError: Error: credential propagation was unsuccessful

<input id="exporta_mapa" class="btn" download="Normales Climatológicas.png" value="Descargar Mapa" type="button">